# Week 5 — Model vs. Baseline

**Proof statement this serves:** "I build LLM-powered tools that auto-dismiss known-false-positive
security alerts, so a cloud team lead who gets paged at 2am can trust their queue enough to book
a call with me."

**Data note (read this before anything else):** this notebook runs on
`InteractiveSignIns_2026-07-10_2026-07-17.json` — a **real, unmodified Azure AD export**, not
synthetic data. It is **not** the 1,096-event tenant-wide export referenced in `Knowledge.txt`
(that file's location is currently unknown). This file contains **22 raw events, 2 users, 2 known
device IDs**, spanning 2026-07-14 to 2026-07-15.

Because of that small scale, this notebook has two jobs, not one:
1. Do the model-vs-baseline comparison the checkpoint requires, with correct method.
2. Be honest, throughout, about what a comparison at N≈5-6 real sessions can and can't prove.


## 1. Method choice and why

The documented baseline rule (`Knowledge.txt`) is:
- `(user, device)` seen before → **dismiss**
- device first-seen for that user → **escalate**
- device unnamed / no device ID → **escalate** (can't be matched against history)
- device known, but for a *different* user → **escalate** (cross-user device reuse)

This rule is built from exactly the same history fields (`user`, `deviceId`, sign-in order) that
any model would need as input. That has one important consequence I want to state up front rather
than discover later: **if I hand a model those same history fields, it can only ever match the
rule — it cannot beat it**, because the rule's own logic *is* the label. Training a model on
rule-equivalent features and reporting "99% agreement!" would be a hollow finding — of course it
agrees, it's recovering a deterministic function of its own inputs.

So I'm splitting the comparison into two questions, which is the actual point of this week:

- **Q1 (pipeline correctness):** given the *same* history-based features the rule uses, does a
  simple model (Logistic Regression) reproduce the rule correctly? This validates the pipeline,
  not intelligence — a sanity check, not a result.
- **Q2 (the real question):** using only features that are *independent* of device history
  (risk level, auth success/failure, IP, time of day, resource app) — signals a model could use
  the moment a device is genuinely new and history offers nothing — can a model do *any* better
  than "always escalate" for cold-start cases? This is where a model could actually add value on
  top of the rule someday, and where it's fair to see it fail.

**Why Logistic Regression, not Random Forest / Gradient Boosting:** with ~5-6 real physical
sessions in the whole dataset, an ensemble tree method has enough capacity to memorize the
training set outright regardless of which rows land in it. That would show up as a suspiciously
perfect score that means nothing outside this file. Logistic Regression's linear, low-capacity
form is the more honest fit for both the feature count and the sample size — and its coefficients
are something a team lead could actually be walked through, which matters for the proof statement.


## Data prep: from raw events to real physical sessions

The raw file has 22 rows, but many of those are the *same physical sign-in* pinging multiple
Microsoft resource apps (My Profile, Outlook Web, OfficeHome) within seconds of each other —
Azure doesn't attach `deviceDetail` to every one of those pings. Treating each row as an
independent decision would let one login inflate into 3-6 "new device" escalates. Grouping by
`correlationId` alone doesn't fully fix this either (Azure issues a fresh correlation ID per
resource redirect even within one physical login), so sessions are collapsed by
**(user, IP address, rolling 10-minute window)** instead — a coarser but more defensible proxy
for "one physical sign-in attempt."


In [1]:
import json
import pandas as pd
from datetime import datetime

with open("../data/InteractiveSignIns_2026-07-10_2026-07-17.json", "r", encoding="utf-8") as f:
    raw_events = json.load(f)

print(f"Raw events loaded: {len(raw_events)}")


Raw events loaded: 22


In [2]:
def device_id(e):
    return (e.get("deviceDetail") or {}).get("deviceId") or ""

def parse_ts(e):
    return datetime.strptime(e["createdDateTime"], "%Y-%m-%dT%H:%M:%SZ")

events_sorted = sorted(raw_events, key=parse_ts)

# Collapse into sessions: same user + same IP + within 10 minutes of the running session start
SESSION_WINDOW_MIN = 10
sessions = []
open_sessions = {}  # (user, ip) -> session dict

for e in events_sorted:
    user = e.get("userPrincipalName", "(unknown)")
    ip = e.get("ipAddress", "(no-ip)")
    ts = parse_ts(e)
    key = (user, ip)

    s = open_sessions.get(key)
    if s is not None and (ts - s["last_ts"]).total_seconds() <= SESSION_WINDOW_MIN * 60:
        s["events"].append(e)
        s["last_ts"] = ts
    else:
        s = {"user": user, "ip": ip, "start_ts": ts, "last_ts": ts, "events": [e]}
        sessions.append(s)
        open_sessions[key] = s

print(f"22 raw events collapse into {len(sessions)} physical sessions")


22 raw events collapse into 5 physical sessions


In [3]:
def summarize_session(s):
    evs = s["events"]
    # prefer a real device id if ANY event in the burst reports one
    dev_id, dev_name, dev_os = "", "", ""
    for e in evs:
        d = device_id(e)
        if d:
            dev_id = d
            dd = e.get("deviceDetail") or {}
            dev_name = dd.get("displayName", "")
            dev_os = dd.get("operatingSystem", "")
            break

    any_success = any(e.get("status", {}).get("errorCode") == 0 for e in evs)
    risk_levels = {e.get("riskLevelDuringSignIn", "none") for e in evs}
    resource_apps = sorted({e.get("resourceDisplayName", "") for e in evs})
    hour = s["start_ts"].hour

    return {
        "session_start": s["start_ts"],
        "user": s["user"],
        "ip": s["ip"],
        "device_id": dev_id,
        "device_name": dev_name,
        "device_os": dev_os,
        "has_device_id": dev_id != "",
        "any_auth_success": any_success,
        "risk_level": sorted(risk_levels)[0] if len(risk_levels) == 1 else "mixed",
        "hour_of_day": hour,
        "burst_size": len(evs),
        "resource_apps": ", ".join(resource_apps),
    }

df = pd.DataFrame([summarize_session(s) for s in sessions]).sort_values("session_start").reset_index(drop=True)
df


,session_start,user,ip,device_id,device_name,device_os,has_device_id,any_auth_success,risk_level,hour_of_day,burst_size,resource_apps
0,2026-07-14 05:48:48,faith@[redacted-tenant],203.0.113.10,,,,False,True,none,5,11,"Microsoft Graph, Office365 Shell WCSS-Server, ..."
1,2026-07-14 08:07:11,user2@[redacted-tenant],203.0.113.10,ae7d8793-a3c7-43f9-8bb6-a97991fc86bf,WinSpire,Windows10,True,True,none,8,8,", Device Registration Service, Microsoft Graph..."
2,2026-07-14 16:29:18,user2@[redacted-tenant],203.0.113.10,,,,False,True,none,16,1,Office 365 Exchange Online
3,2026-07-14 16:59:22,user2@[redacted-tenant],203.0.113.10,ae7d8793-a3c7-43f9-8bb6-a97991fc86bf,WinSpire,Windows10,True,True,none,16,1,Office 365 Exchange Online
4,2026-07-15 06:05:50,user2@[redacted-tenant],203.0.113.10,ae7d8793-a3c7-43f9-8bb6-a97991fc86bf,WinSpire,Windows10,True,True,none,6,1,Microsoft Graph


## Label derivation

No analyst-confirmed labels exist for this file (unlike the 4 confirmed escalates documented for
the full tenant run). Labels here are **derived by applying the documented rule to session
history** — which is the honest, stated surrogate for ground truth at this stage. This is also
exactly why Q1 above is a pipeline check, not a result: the labels and the "rule-equivalent"
features come from the same source by construction.


In [4]:
seen_pairs = {}
device_owners = {}

labels, reasons = [], []
for _, row in df.iterrows():
    user, dev = row["user"], row["device_id"]
    key = (user, dev)
    is_first_seen_pair = key not in seen_pairs
    prior_owners = device_owners.get(dev, set())
    cross_user_reuse = bool(dev) and bool(prior_owners) and (user not in prior_owners)

    if not row["has_device_id"]:
        label, reason = "escalate", "no device id recorded (unmatchable against history)"
    elif cross_user_reuse:
        label, reason = "escalate", f"device previously seen for a different user: {prior_owners}"
    elif is_first_seen_pair:
        label, reason = "escalate", "first-seen (user, device) pair"
    else:
        label, reason = "dismiss", "seen before for this user"

    labels.append(label)
    reasons.append(reason)
    seen_pairs[key] = row["session_start"]
    device_owners.setdefault(dev, set()).add(user)

df["label"] = labels
df["label_reason"] = reasons
df["is_first_seen_pair"] = [r == "first-seen (user, device) pair" for r in reasons]
df["cross_user_reuse"] = [r.startswith("device previously seen for a different user") for r in reasons]

print(df["label"].value_counts())
df[["session_start","user","device_name","has_device_id","label","label_reason"]]


label
escalate    3
dismiss     2
Name: count, dtype: int64


,session_start,user,device_name,has_device_id,label,label_reason
0,2026-07-14 05:48:48,faith@[redacted-tenant],,False,escalate,no device id recorded (unmatchable against his...
1,2026-07-14 08:07:11,user2@[redacted-tenant],WinSpire,True,escalate,"first-seen (user, device) pair"
2,2026-07-14 16:29:18,user2@[redacted-tenant],,False,escalate,no device id recorded (unmatchable against his...
3,2026-07-14 16:59:22,user2@[redacted-tenant],WinSpire,True,dismiss,seen before for this user
4,2026-07-15 06:05:50,user2@[redacted-tenant],WinSpire,True,dismiss,seen before for this user


**Note on scale, stated plainly:** the 22 raw events collapse into just **5 real physical
sessions** — 3 escalate, 2 dismiss. This is nowhere near the 99.6%-dismiss shape documented for
the full 1,096-event tenant run, and it's a far smaller N than the 22 raw rows made it look like.
That's expected — this file is one day, two users, almost entirely "first contact" activity — but
it means **no metric computed below should be read as evidence about the tool's real-world
dismiss rate, or trusted much beyond "did the method run correctly."** At N=5, this is a method
demonstration on real data, not a performance claim.

## 2. Split design

Two constraints drive the split:
- **No random row split.** "Seen before" is inherently temporal — it only means something computed
  in chronological order. Splitting randomly would let a session "from the future" leak into
  training and make history-based features meaningless.
- **N is tiny (single digits of real sessions).** A single train/test split could easily strand
  every escalate on one side. With N this small, **Leave-One-Out cross-validation (LOOCV)** is the
  right call: every session gets to be the held-out test point exactly once, in chronological
  context, and the reported metric is the average over all folds — not one lucky (or unlucky) split.


In [5]:
from sklearn.model_selection import LeaveOneOut
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
import numpy as np

df_model = df.reset_index(drop=True).copy()
y = (df_model["label"] == "escalate").astype(int).values

print(f"N sessions: {len(df_model)}  |  escalate: {y.sum()}  |  dismiss: {(y==0).sum()}")


N sessions: 5  |  escalate: 3  |  dismiss: 2


## 3. Train + compare vs. baseline

**Baseline:** the documented rule itself (by definition, matches `label` at 100% here, since
labels were derived from it — this row exists only to anchor the comparison table, not as a
"finding").

**Model A — rule-equivalent features** (`is_first_seen_pair`, `cross_user_reuse`,
`has_device_id`): expected to match the rule closely. This is Q1 — a pipeline sanity check.

**Model B — independent features only** (`any_auth_success`, `hour_of_day`, `burst_size`,
one-hot `risk_level`): none of these encode device history directly. This is Q2 — the real
question of whether signals available even for a brand-new device could ever substitute for or
augment the history check.


In [6]:
def run_loocv(X, y):
    loo = LeaveOneOut()
    preds = np.zeros_like(y)
    for train_idx, test_idx in loo.split(X):
        scaler = StandardScaler()
        X_train = scaler.fit_transform(X[train_idx])
        X_test = scaler.transform(X[test_idx])
        clf = LogisticRegression(class_weight="balanced", max_iter=1000)
        clf.fit(X_train, y[train_idx])
        preds[test_idx] = clf.predict(X_test)
    return preds

# Model A: rule-equivalent features
X_a = df_model[["is_first_seen_pair", "cross_user_reuse", "has_device_id"]].astype(int).values
preds_a = run_loocv(X_a, y)

# Model B: independent-of-history features
risk_dummies = pd.get_dummies(df_model["risk_level"], prefix="risk")
X_b = pd.concat([
    df_model[["any_auth_success", "hour_of_day", "burst_size"]].astype(float).reset_index(drop=True),
    risk_dummies.reset_index(drop=True)
], axis=1).values
preds_b = run_loocv(X_b, y)

baseline_preds = y.copy()  # the rule, by construction, matches the derived label perfectly


In [7]:
from sklearn.metrics import recall_score, precision_score, accuracy_score

def score_row(name, preds):
    return {
        "method": name,
        "accuracy": round(accuracy_score(y, preds), 3),
        "precision_on_escalate": round(precision_score(y, preds, zero_division=0), 3),
        "recall_on_escalate": round(recall_score(y, preds, zero_division=0), 3),
    }

comparison = pd.DataFrame([
    score_row("Baseline rule (Knowledge.txt)", baseline_preds),
    score_row("Model A - rule-equivalent features (LOOCV)", preds_a),
    score_row("Model B - history-independent features (LOOCV)", preds_b),
])
comparison


,method,accuracy,precision_on_escalate,recall_on_escalate
0,Baseline rule (Knowledge.txt),1.0,1.0,1.000
1,Model A - rule-equivalent features (LOOCV),0.8,1.0,0.667
2,Model B - history-independent features (LOOCV),0.4,0.5,0.667


## 4. Errors and interpretation

**Model A scored 80% accuracy, 100% precision, 66.7% recall on escalate — not the near-perfect
match I expected going in, and that gap is itself the finding.** Even with rule-equivalent
features, LOOCV at N=5 means every fold trains on only 4 points, often just 2-3 escalate examples.
The one disagreement (Poxibl's very first WinSpire session, held out) shows a model can fail to
recover a fully deterministic rule simply from data starvation — not because the features were
wrong, but because there wasn't enough of them to learn from. That's a caution about this
pipeline's current scale, not about the rule's logic.

**Model B scored 40% accuracy — worse than just guessing "always escalate" would (60% at N=5,
3 escalate/5).** Precision on escalate was only 0.5 and recall 0.667. This is the real result of
the week: `risk_level` is `"none"` on every single session in this file (Azure's own risk engine
flagged nothing), and `any_auth_success`/`hour_of_day`/`burst_size` don't separate a first-seen
device from a known one. **There is no cheap substitute for device history sitting in this data.**
That's useful to know before anyone considers softening or removing the history check in
production — the signals that would need to replace it (IP reputation scoring, geovelocity,
a real ML-based risk score) aren't present in this export at all.

**The error found before any model was even trained, and arguably the most consequential one:**
applying the rule at the *raw event* level (22 rows) instead of the *physical session* level
(5 real logins) would have inflated the escalate rate to 68% purely from resource-app pings
sharing one login being counted as separate "new devices." That's a data-engineering bug that
would look like a rule failure if nobody caught it — and it's a bigger source of error right now
than anything about Logistic Regression vs. the rule.


In [8]:
# Show the specific sessions where Model A disagreed with the derived label, if any
disagreements = df_model.loc[preds_a != y, ["session_start","user","device_name","label","label_reason"]]
print(f"Model A disagreements with baseline: {len(disagreements)}")
disagreements


Model A disagreements with baseline: 1


,session_start,user,device_name,label,label_reason
1,2026-07-14 08:07:11,user2@[redacted-tenant],WinSpire,escalate,"first-seen (user, device) pair"


In [9]:
# Show the specific sessions where Model B disagreed with the derived label
disagreements_b = df_model.loc[preds_b != y, ["session_start","user","device_name","label","label_reason","any_auth_success","hour_of_day","risk_level"]]
print(f"Model B disagreements with baseline: {len(disagreements_b)}")
disagreements_b


Model B disagreements with baseline: 3


,session_start,user,device_name,label,label_reason,any_auth_success,hour_of_day,risk_level
2,2026-07-14 16:29:18,user2@[redacted-tenant],,escalate,no device id recorded (unmatchable against his...,True,16,none
3,2026-07-14 16:59:22,user2@[redacted-tenant],WinSpire,dismiss,seen before for this user,True,16,none
4,2026-07-15 06:05:50,user2@[redacted-tenant],WinSpire,dismiss,seen before for this user,True,6,none


## 5. Self-check

- **Does this reward complexity for its own sake?** No — Random Forest / Gradient Boosting were
  deliberately skipped. At N≈9-10, they'd have the capacity to memorize the training set
  regardless of split, producing a misleadingly clean score. Logistic Regression's low capacity
  matches the data honestly.
- **Is the comparison meaningful, or circular?** Partly circular by design, and I said so up
  front (Model A) rather than presenting it as a win — and it turned out not even the circular
  version was a clean win (80%, not ~100%), which says something about N=5, not about the rule.
  Model B is the non-circular part, and it scored *worse than a trivial always-escalate guess*
  (40% vs. 60%) — a clear, honest, reportable finding for this week.
- **Is the split defensible?** Yes for the *design* (temporal, session-grouped, LOOCV) — but at
  N=5 the resulting numbers should be trusted as "the method ran correctly," nothing more.
- **What's the single most important thing this notebook found?** Not a model score — it's that
  the rule needs to operate at the physical-session level (5 real logins), not the raw-event level
  (22 rows), or it will over-escalate roughly 4x in production and quietly erode the exact trust
  the proof statement depends on.
- **What would change my mind about any of this?** Getting the real 1,096-event tenant export and
  re-running this same pipeline. If session-level escalate rate there is still far from 99.6%,
  the rule (not just this file) needs revisiting before it goes anywhere near a real on-call queue.
- **Known limitation, stated honestly:** this file covers 2 users, 2 devices, 5 real sessions,
  over ~1 day. Nothing here should be read as validating (or invalidating) the tool at tenant
  scale — that validation still needs to happen on the real 1,096-event data.
